In [1]:
rlef_model = "678a262dc441e0b2c81a9686"


from vdeo_analysis_ellm_sudio import VideoAnalyzer


import json
import os

from utils import get_signed_url, upload_hdf5_file
from rlef_video_annotation import VideoUploader


with open('payload.json','r') as file:
    payload = json.load(file)


video_analyzer = VideoAnalyzer(payload=payload)
records_path = 'completed'


def send_all_recordings_to_rlef(recording_dir=records_path):
    folders = os.listdir(recording_dir)
    for folder in folders:
        subfolder_path = f'{recording_dir}/{folder}'

        if not os.path.isdir(subfolder_path):
            continue

        csv_path =f'{subfolder_path}/predictions_hamer.csv'
        if not os.path.exists(csv_path):
            print(f'Skipping {subfolder_path}: csv predictions NOT FOUND')
            continue

        print(f"================= PROCESSING {subfolder_path} =================")
        url = "https://autoai-backend-exjsxe2nda-uc.a.run.app/resource/"
        video_filepath =f'{subfolder_path}/color.mp4'
        if not os.path.exists(video_filepath):
            print(f'Skipping {subfolder_path}/color.mp4: csv predictions NOT FOUND')
            continue
        gcp_url = video_analyzer.upload_video_to_bucket('test1.mp4',video_file_path=video_filepath)
        video_annotations = video_analyzer.get_gemini_response(gcp_url=gcp_url)
        uploader = VideoUploader(video_filepath, video_annotations=video_annotations)
        status_code, rlef_response_text = uploader.upload_to_rlef_train(url, video_filepath, video_annotations, rlef_model)
        print(f"================== STATUS CODE ============ \n{status_code}")
        print(f"\nUPDATING WITH THE CSV FILES\n")
        signed_url = get_signed_url(rlef_response_text['_id'], "predictions_hamer.csv")
        if signed_url:
            print(f"SIGNED URL: {signed_url}")
            upload_hdf5_file(signed_url, f'{csv_path}')
        
            print('================================ UPLOADED CSV =========================================')

In [ ]:
send_all_recordings_to_rlef(recording_dir=records_path)

================= PROCESSING completed/20250114_145437 =================
{'overall_task_name': 'placing a soda can next to a mug', 'objects': ['soda can', 'mug'], 'picking up': [{'start_time': '00:00', 'end_time': '00:06', 'object_name': 'soda can', 'notes': 'The human picks up the soda can.'}], 'placing': [{'start_time': '00:06', 'end_time': '00:07', 'object_name': 'soda can', 'notes': 'The human places the soda can next to the mug.'}]}
{'overall_task_name': 'placing a soda can next to a mug', 'objects': ['soda can', 'mug'], 'picking up': [{'start_time': '00:00', 'end_time': '00:06', 'object_name': 'soda can', 'notes': 'The human picks up the soda can.'}], 'placing': [{'start_time': '00:06', 'end_time': '00:07', 'object_name': 'soda can', 'notes': 'The human places the soda can next to the mug.'}]}
RLEF RESPONSE STATUS: =========== 200
================== STATUS CODE ============ 
200

UPDATING WITH THE CSV FILES

SIGNED URL: https://storage.googleapis.com/auto-ai_resources_fo/project_